In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    FeatureHasher
)
from pyspark.ml import Pipeline

import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import QuantileTransformer

### Load Raw Tables

In [0]:
# DATA_ROOT = "s3://university-research-s20426"

# cg  = spark.read.parquet(f"{DATA_ROOT}/MSCallGraph_clean")
# res = spark.read.parquet(f"{DATA_ROOT}/resource")
# rtq = spark.read.parquet(f"{DATA_ROOT}/MSRTQps_clean")

In [0]:
d_res = res.withColumn("t_idx", (F.col("timestamp") / 60000).cast("int"))
d_cg = cg.withColumn("t_idx", (F.col("timestamp") / 60000).cast("int"))
d_rtq = rtq.withColumn("t_idx", (F.col("timestamp") / 60000).cast("int"))

In [0]:
print("resource table :t_idx count:",d_res.select("t_idx").distinct().count())
print("mrtqps table :t_idx count:",d_rtq.select("t_idx").distinct().count())
print("cg table :t_idx count:",d_cg.select("t_idx").distinct().count())

resource table :t_idx count: 720
mrtqps table :t_idx count: 339
cg table :t_idx count: 721


### Temporal SPlit

In [0]:
TRAIN_END = 650
VAL_END   =700
TEST_END=720

# =========================
# RESOURCE TABLE
# =========================

res_train = res.filter(F.col("t_idx") < TRAIN_END)
res_val = res.filter(
    (F.col("t_idx") >= TRAIN_END) &
    (F.col("t_idx") < VAL_END)
)
res_test = res.filter(
    (F.col("t_idx") >= VAL_END) &
   ( F.col("t_idx") <= TEST_END)
)

# =========================
# RTQPS TABLE
# =========================

rtq_train = rtq.filter(F.col("t_idx") < TRAIN_END)
rtq_val = rtq.filter(
    (F.col("t_idx") >= TRAIN_END) &
    (F.col("t_idx") < VAL_END)
)
rtq_test = rtq.filter( 
    (F.col("t_idx") >= VAL_END) &
   ( F.col("t_idx") <= TEST_END)
)
# =========================
# CALL GRAPH TABLE
# =========================

cg_train = cg.filter(F.col("t_idx") < TRAIN_END)
cg_val = cg.filter(
    (F.col("t_idx") >= TRAIN_END) &
    (F.col("t_idx") < VAL_END)
)
cg_test = cg.filter( 
    (F.col("t_idx") >= VAL_END) &
   ( F.col("t_idx") <= TEST_END)
)

### Helper Functions

In [0]:

def compute_robust_stats(df, col_name):
    quantiles = df.approxQuantile(
        col_name,
        [0.25, 0.5, 0.75],
        0.01
    )

    q1, q2, q3 = quantiles
    iqr = q3 - q1

    if iqr < 1e-8:
        iqr = 1.0

    return q1, q2, q3, iqr



def robust_scale_column(df, col_name, q2, iqr, output_col=None):

    if output_col is None:
        output_col = f"{col_name}_rs"

    return df.withColumn(
        output_col,
        (F.col(col_name) - F.lit(q2)) / F.lit(iqr)
    )

## Resource Feature Engineering

In [0]:
cpu_q1, cpu_q2, cpu_q3, cpu_iqr = compute_robust_stats(
    res_train,
    "cpu_utilization"
)

mem_q1, mem_q2, mem_q3, mem_iqr = compute_robust_stats(
    res_train,
    "memory_utilization"
)

In [0]:

def transform_resource(df):

    df = robust_scale_column(
        df,
        "cpu_utilization",
        cpu_q2,
        cpu_iqr,
        "cpu_rs"
    )

    df = robust_scale_column(
        df,
        "memory_utilization",
        mem_q2,
        mem_iqr,
        "memory_rs"
    )

    return df


res_train = transform_resource(res_train)
res_val = transform_resource(res_val)
res_test = transform_resource(res_test)

## Rolling Temporal Features

In [0]:
window_5 = (
    Window
    .partitionBy("msinstanceid")
    .orderBy("t_idx")
    .rowsBetween(-4, 0)
)

lag_window = (
    Window
    .partitionBy("msinstanceid")
    .orderBy("t_idx")
)

In [0]:

def add_temporal_features(df):

    # =========================
    # CPU FEATURES
    # =========================

    df = df.withColumn(
        "cpu_rolling_mean",
        F.avg("cpu_rs").over(window_5)
    )

    df = df.withColumn(
        "cpu_rolling_std",
        F.stddev("cpu_rs").over(window_5)
    )

    df = df.withColumn(
        "cpu_lag1",
        F.lag("cpu_rs", 1).over(lag_window)
    )

    df = df.withColumn(
        "cpu_delta",
        F.when(F.col("cpu_lag1").isNull(), 0.0)
        .otherwise(F.col("cpu_rs") - F.col("cpu_lag1"))
    )

    # =========================
    # MEMORY FEATURES
    # =========================

    df = df.withColumn(
        "memory_rolling_mean",
        F.avg("memory_rs").over(window_5)
    )

    df = df.withColumn(
        "memory_rolling_std",
        F.stddev("memory_rs").over(window_5)
    )

    df = df.withColumn(
        "memory_lag1",
        F.lag("memory_rs", 1).over(lag_window)
    )

    df = df.withColumn(
        "memory_delta",
        F.when(F.col("memory_lag1").isNull(), 0.0)
        .otherwise(F.col("memory_rs") - F.col("memory_lag1"))
    )

    return df.drop("cpu_lag1", "memory_lag1")

In [0]:
res_train = add_temporal_features(res_train)
res_val = add_temporal_features(res_val)
res_test = add_temporal_features(res_test)

## Cyclic Time Encoding

In [0]:
PERIOD = TRAIN_END


def add_time_encoding(df):

    df = df.withColumn(
        "t_sin",
        F.sin(2 * np.pi * F.col("t_idx") / PERIOD)
    )

    df = df.withColumn(
        "t_cos",
        F.cos(2 * np.pi * F.col("t_idx") / PERIOD)
    )

    return df


res_train = add_time_encoding(res_train)
res_val = add_time_encoding(res_val)
res_test = add_time_encoding(res_test)

## MRTQps Feature Engineering

### Replace Invalid RT zeros

In [0]:
rt_cols = [c for c in rtq.columns if c.endswith("_RT")]


def fix_rt_zeros(df):

    for col in rt_cols:
        df = df.withColumn(
            col,
            F.when(F.col(col) == 0, 0.001)
            .otherwise(F.col(col))
        )

    return df


rtq_train = fix_rt_zeros(rtq_train)
rtq_val = fix_rt_zeros(rtq_val)
rtq_test = fix_rt_zeros(rtq_test)

### Log Transform

In [0]:
mcr_cols = [c for c in rtq.columns if c.endswith("_MCR")]
all_traffic_cols = mcr_cols + rt_cols


def add_log_features(df):

    for col in all_traffic_cols:

        df = df.withColumn(
            f"{col}_log1p",
            F.log1p(F.col(col))
        )

    return df


rtq_train = add_log_features(rtq_train)
rtq_val = add_log_features(rtq_val)
rtq_test = add_log_features(rtq_test)

### Fit Scaling only on trian

In [0]:
traffic_stats = {}

for col in all_traffic_cols:

    log_col = f"{col}_log1p"

    q1, q2, q3, iqr = compute_robust_stats(
        rtq_train,
        log_col
    )

    traffic_stats[col] = {
        'q2': q2,
        'iqr': iqr
    }

### Apply Scaling everywhere

In [0]:

def transform_rtq(df):

    for col in all_traffic_cols:

        log_col = f"{col}_log1p"

        q2 = traffic_stats[col]['q2']
        iqr = traffic_stats[col]['iqr']

        df = robust_scale_column(
            df,
            log_col,
            q2,
            iqr,
            output_col=f"{col}_transformed"
        )

    return df


rtq_train = transform_rtq(rtq_train)
rtq_val = transform_rtq(rtq_val)
rtq_test = transform_rtq(rtq_test)

## Call Graph Feature Engineering

### Quantile Transformer

In [0]:
sample_size = min(1_000_000, cg_train.count())

cg_sample_pd = (
    cg_train
    .select("rt", "rt_abs")
    .sample(
        withReplacement=False,
        fraction=sample_size / cg_train.count(),
        seed=42
    )
    .toPandas()
)

### Fit Quantile Transformers

In [0]:
qt_rt = QuantileTransformer(
    output_distribution='normal',
    n_quantiles=1000
)

qt_rt.fit(cg_sample_pd[['rt']])

qt_rt_abs = QuantileTransformer(
    output_distribution='normal',
    n_quantiles=1000
)

qt_rt_abs.fit(cg_sample_pd[['rt_abs']])

QuantileTransformer(output_distribution='normal')

### Create Spark UDFs

In [0]:
from pyspark.sql.functions import pandas_udf


def make_quantile_udf(qt):

    qt_bytes = pickle.dumps(qt)
    _cache = {}

    @pandas_udf(DoubleType())
    def _udf(s: pd.Series) -> pd.Series:

        import pickle

        if "qt" not in _cache:
            _cache["qt"] = pickle.loads(qt_bytes)

        return pd.Series(
            _cache["qt"]
            .transform(s.values.reshape(-1, 1))
            .flatten()
        )

    return _udf


quantile_transform_rt = make_quantile_udf(qt_rt)
quantile_transform_rt_abs = make_quantile_udf(qt_rt_abs)

### Apply Quantile Transform

In [0]:

def transform_call_graph(df):

    df = df.withColumn(
        "rt_qt",
        quantile_transform_rt(F.col("rt"))
    )

    df = df.withColumn(
        "rt_abs_qt",
        quantile_transform_rt_abs(F.col("rt_abs"))
    )

    return df


cg_train = transform_call_graph(cg_train)
cg_val = transform_call_graph(cg_val)
cg_test = transform_call_graph(cg_test)

### Categorical Encoding


In [0]:
rpctype_indexer = StringIndexer(
    inputCol="rpctype",
    outputCol="rpctype_idx",
    handleInvalid="keep"
)

rpctype_encoder = OneHotEncoder(
    inputCols=["rpctype_idx"],
    outputCols=["rpctype_ohe"]
)

# interface_hasher = FeatureHasher(
#     inputCols=["interface"],
#     outputCol="interface_hashed",
#     numFeatures=2**18
# )

pipeline = Pipeline(stages=[
    rpctype_indexer,
    rpctype_encoder
])

In [0]:
# pipeline_model = pipeline.fit(cg_train)

In [0]:
cg_train = pipeline_model.transform(cg_train)
cg_val = pipeline_model.transform(cg_val)
cg_test = pipeline_model.transform(cg_test)

In [0]:
res.schema

StructType([StructField('msname', StringType(), True), StructField('msinstanceid', StringType(), True), StructField('nodeid', StringType(), True), StructField('cpu_utilization', DoubleType(), True), StructField('memory_utilization', DoubleType(), True), StructField('timestamp', LongType(), True), StructField('t_idx', IntegerType(), True)])

In [0]:
rtq.schema

StructType([StructField('timestamp', LongType(), True), StructField('msname', StringType(), True), StructField('msinstanceid', StringType(), True), StructField('t_idx', IntegerType(), True), StructField('HTTP_MCR', DoubleType(), True), StructField('HTTP_RT', DoubleType(), True), StructField('consumerMQ_MCR', DoubleType(), True), StructField('consumerMQ_RT', DoubleType(), True), StructField('consumerRPC_MCR', DoubleType(), True), StructField('consumerRPC_RT', DoubleType(), True), StructField('providerRPC_MCR', DoubleType(), True), StructField('providerRPC_RT', DoubleType(), True)])

In [0]:
cg.schema

StructType([StructField('traceid', StringType(), True), StructField('timestamp', LongType(), True), StructField('rpcid', StringType(), True), StructField('UM', StringType(), True), StructField('rpctype', StringType(), True), StructField('DM', StringType(), True), StructField('interface', StringType(), True), StructField('rt', DoubleType(), True), StructField('t_idx', IntegerType(), True), StructField('rt_abs', DoubleType(), True), StructField('rt_direction', StringType(), True)])

In [0]:
# !pip install torch

========================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
from pyspark.sql.functions import broadcast
import json
 
# ── CONFIG ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = "s3://university-research-s20426"
OUTPUT_ROOT = f"{DATA_ROOT}/tgnn_dataset"
 
TRAIN_END = 650
VAL_END   = 700
 
# Node feature columns (produced by Feature_Engineering_2)
NODE_RES_FEATURES = [
    "cpu_rs", "cpu_rolling_mean", "cpu_rolling_std", "cpu_delta",
    "memory_rs", "memory_rolling_mean", "memory_rolling_std", "memory_delta",
    "t_sin", "t_cos",
]
 
# RTQ transformed column names (dynamic — built from all_traffic_cols)
RTQ_TRANSFORMED_COLS = [f"{c}_transformed" for c in all_traffic_cols]
 
# All node feature columns combined
NODE_FEATURE_COLS = NODE_RES_FEATURES + RTQ_TRANSFORMED_COLS
 
# Edge feature columns (scalar; rpctype_ohe is a vector and handled separately)
EDGE_FEATURE_COLS = ["rt_qt", "rt_abs_qt"]
 
print(f"Node feature dims : {len(NODE_FEATURE_COLS)}")
print(f"Edge feature dims : {len(EDGE_FEATURE_COLS)}  (+rpctype_ohe vector)")
print(f"RTQ cols          : {len(RTQ_TRANSFORMED_COLS)}")

Node feature dims : 18
Edge feature dims : 2  (+rpctype_ohe vector)
RTQ cols          : 8


In [0]:
all_ids = (
    # Resource instance IDs
    res_train.select(F.col("msinstanceid").alias("node_key"))
    .union(res_val.select(F.col("msinstanceid").alias("node_key")))
    .union(res_test.select(F.col("msinstanceid").alias("node_key")))
    # Call graph upstream (caller) IDs
    .union(cg_train.select(F.col("UM").alias("node_key")))
    .union(cg_val.select(F.col("UM").alias("node_key")))
    .union(cg_test.select(F.col("UM").alias("node_key")))
    # Call graph downstream (callee) IDs
    .union(cg_train.select(F.col("DM").alias("node_key")))
    .union(cg_val.select(F.col("DM").alias("node_key")))
    .union(cg_test.select(F.col("DM").alias("node_key")))
    .distinct()
)
 
node_index = (
    all_ids
    .orderBy("node_key")
    .withColumn("node_id", F.monotonically_increasing_id().cast(LongType()))
)
 
node_index.write.mode("overwrite").parquet(f"{OUTPUT_ROOT}/node_index")
node_index = spark.read.parquet(f"{OUTPUT_ROOT}/node_index")
 
print(f"Total unique nodes: {node_index.count():,}")
node_index.limit(5).show(truncate=False)
 


Total unique nodes: 111,296
+----------------------------------------------------------------+-----------+
|node_key                                                        |node_id    |
+----------------------------------------------------------------+-----------+
|3b1520c6a5b9d46f17ba89e5f01375584ef0e7d40a065d8f0d383de3d736159c|17179869184|
|3b156ec7570fafbbb52aed85ebbfe59db8e2ad78927af8fa085797e886c55cfb|17179869185|
|3b1585f12fc0b9ffff68777104a2d2578ece5cf96961a70533911ea2fc6058ca|17179869186|
|3b15872992a9a704d743311e20a9d84262db7c940a984225b1c212418b282292|17179869187|
|3b160adb93f08a880a55b980a9b89fb9bb6b7ce0cc4e870965f4cb4be2750b09|17179869188|
+----------------------------------------------------------------+-----------+



In [0]:
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## Step 2 — Verify Node Index Coverage
 
# COMMAND ----------
 
node_lookup = spark.read.parquet(f"{OUTPUT_ROOT}/node_index")
 
um_count  = cg_train.select("UM").distinct().count()
dm_count  = cg_train.select("DM").distinct().count()
res_count = res_train.select("msinstanceid").distinct().count()
 
um_hits = (
    cg_train.select(F.col("UM").alias("node_key")).distinct()
    .join(node_lookup, on="node_key", how="inner").count()
)
dm_hits = (
    cg_train.select(F.col("DM").alias("node_key")).distinct()
    .join(node_lookup, on="node_key", how="inner").count()
)
res_hits = (
    res_train.select(F.col("msinstanceid").alias("node_key")).distinct()
    .join(node_lookup, on="node_key", how="inner").count()
)
 
print(f"UM  coverage : {um_hits:,} / {um_count:,}")
print(f"DM  coverage : {dm_hits:,} / {dm_count:,}")
print(f"res coverage : {res_hits:,} / {res_count:,}")
 

UM  coverage : 3,092 / 3,092
DM  coverage : 14,595 / 14,595
res coverage : 96,392 / 96,392


In [0]:
def build_edge_features(cg_df, split_name):

    node_lookup = spark.read.parquet(f"{OUTPUT_ROOT}/node_index")

    # src: join UM against node_key, alias node_id as src_node_id
    cg_with_src = (
        cg_df
        .join(
            broadcast(node_lookup.select(
                F.col("node_key"),
                F.col("node_id").alias("src_node_id")
            )),
            on=cg_df["UM"] == node_lookup["node_key"],
            how="left"
        )
        .drop("node_key")
    )

    # dst: join DM against node_key, alias node_id as dst_node_id
    # Re-read node_lookup fresh to avoid lineage conflicts
    node_lookup2 = spark.read.parquet(f"{OUTPUT_ROOT}/node_index")

    cg_with_dst = (
        cg_with_src
        .join(
            broadcast(node_lookup2.select(
                F.col("node_key"),
                F.col("node_id").alias("dst_node_id")
            )),
            on=cg_with_src["DM"] == node_lookup2["node_key"],
            how="left"
        )
        .drop("node_key")
    )

    edge_feat = cg_with_dst.select(
        F.col("src_node_id"),
        F.col("dst_node_id"),
        F.col("t_idx"),
        F.col("traceid"),
        F.col("rpctype"),
        *[F.col(c) for c in EDGE_FEATURE_COLS],
        F.col("rpctype_ohe").cast("string").alias("rpctype_ohe_str"),
    ).filter(
        F.col("src_node_id").isNotNull() |
        F.col("dst_node_id").isNotNull()
    )

    edge_feat.write.mode("overwrite").parquet(
        f"{OUTPUT_ROOT}/edge_features/{split_name}"
    )

    count = spark.read.parquet(f"{OUTPUT_ROOT}/edge_features/{split_name}").count()
    print(f"[{split_name}] edge features: {count:,} rows")
    return spark.read.parquet(f"{OUTPUT_ROOT}/edge_features/{split_name}")


edge_feat_train = build_edge_features(cg_train, "train")
edge_feat_val   = build_edge_features(cg_val,   "val")
edge_feat_test  = build_edge_features(cg_test,  "test")

[train] edge features: 149,242,175 rows
[val] edge features: 19,334,386 rows
[test] edge features: 9,345,111 rows


In [0]:
# MAGIC ## Step 5 — Verify Edge Coverage Before Snapshot Write
 
# COMMAND ----------
 
ef = spark.read.parquet(f"{OUTPUT_ROOT}/edge_features/train")
 
print("Edge feature row count   :", ef.count())
print("Null src_node_id         :", ef.filter(F.col("src_node_id").isNull()).count())
print("Null dst_node_id         :", ef.filter(F.col("dst_node_id").isNull()).count())
print("Distinct t_idx values    :", ef.select("t_idx").distinct().count())
 
ef.select("src_node_id", "dst_node_id", "t_idx", *EDGE_FEATURE_COLS).limit(5).show()

Edge feature row count   : 149242175
Null src_node_id         : 0
Null dst_node_id         : 0
Distinct t_idx values    : 650
+-----------+-----------+-----+-------------------+------------------+
|src_node_id|dst_node_id|t_idx|              rt_qt|         rt_abs_qt|
+-----------+-----------+-----+-------------------+------------------+
|60129553273|34359738550|   64|  1.952047608957102|1.5336120376100353|
|60129546286|60129557560|  159|-0.3789773215364034|-5.199337582605575|
|34359738550|          0|  114|  1.044408794872596| 0.526416867571128|
| 8589935616|17179875988|  207| 2.2816611834116163|1.8779081808027296|
|42949674144|       4738|  177|  1.102440367015165|0.5939395388892443|
+-----------+-----------+-----+-------------------+------------------+



In [0]:
def write_snapshots(node_feat_df, edge_feat_df, split_name):

    # ── Node snapshot ──────────────────────────────────────────────────────────
    node_cols = [c for c in node_feat_df.columns if c != "t_idx"]

    (
        node_feat_df
        .select("t_idx", *node_cols)
        .write
        .mode("overwrite")
        .partitionBy("t_idx")
        .option("compression", "snappy")
        .parquet(f"{OUTPUT_ROOT}/snapshots/{split_name}/nodes")
    )

    # ── Edge snapshot ──────────────────────────────────────────────────────────
    # rpctype_ohe_str is already the string-cast version — select all cols except t_idx
    edge_cols = [c for c in edge_feat_df.columns if c != "t_idx"]

    (
        edge_feat_df
        .select("t_idx", *edge_cols)
        .write
        .mode("overwrite")
        .partitionBy("t_idx")
        .option("compression", "snappy")
        .parquet(f"{OUTPUT_ROOT}/snapshots/{split_name}/edges")
    )

    # ── Verify ─────────────────────────────────────────────────────────────────
    node_parts = dbutils.fs.ls(f"{OUTPUT_ROOT}/snapshots/{split_name}/nodes")
    edge_parts = dbutils.fs.ls(f"{OUTPUT_ROOT}/snapshots/{split_name}/edges")

    n_node_parts = sum(1 for p in node_parts if "t_idx=" in p.name)
    n_edge_parts = sum(1 for p in edge_parts if "t_idx=" in p.name)

    print(f"[{split_name}] node snapshots: {n_node_parts} partitions")
    print(f"[{split_name}] edge snapshots: {n_edge_parts} partitions")


write_snapshots(node_feat_train, edge_feat_train, "train")
write_snapshots(node_feat_val,   edge_feat_val,   "val")
write_snapshots(node_feat_test,  edge_feat_test,  "test")

[train] node snapshots: 650 partitions
[train] edge snapshots: 650 partitions
[val] node snapshots: 50 partitions
[val] edge snapshots: 50 partitions
[test] node snapshots: 740 partitions
[test] edge snapshots: 741 partitions


In [0]:
def write_manifest(node_feat_df, split_name):

    t_indices = (
        node_feat_df
        .select("t_idx")
        .distinct()
        .orderBy("t_idx")
        .toPandas()["t_idx"]
        .astype(int)
        .tolist()
    )

    manifest = {
        "split":      split_name,
        "t_indices":  t_indices,
        "count":      len(t_indices),
        "nodes_path": f"{OUTPUT_ROOT}/snapshots/{split_name}/nodes",
        "edges_path": f"{OUTPUT_ROOT}/snapshots/{split_name}/edges",
    }

    mdf = spark.createDataFrame([(json.dumps(manifest, indent=2),)], ["json"])
    mdf.coalesce(1).write.mode("overwrite").text(
        f"{OUTPUT_ROOT}/manifests/{split_name}"
    )

    print(f"[{split_name}] manifest: {len(t_indices)} snapshots  "
          f"(t={t_indices[0]} → t={t_indices[-1]})")
    return t_indices


train_t = write_manifest(node_feat_train, "train")
val_t   = write_manifest(node_feat_val,   "val")
test_t  = write_manifest(node_feat_test,  "test")

[train] manifest: 650 snapshots  (t=0 → t=649)
[val] manifest: 50 snapshots  (t=650 → t=699)
[test] manifest: 740 snapshots  (t=700 → t=1439)


In [0]:
stats = {
    "num_nodes":           node_index.count(),
    "num_node_features":   len(NODE_FEATURE_COLS),
    "num_edge_features":   len(EDGE_FEATURE_COLS),
    "node_feature_names":  NODE_FEATURE_COLS,
    "edge_feature_names":  EDGE_FEATURE_COLS,
    "train_t_range":       [int(train_t[0]),  int(train_t[-1])],
    "val_t_range":         [int(val_t[0]),    int(val_t[-1])],
    "test_t_range":        [int(test_t[0]),   int(test_t[-1])],
    "s3_root":             OUTPUT_ROOT,
    "notes": {
        "rpctype_ohe": "Stored as string in edge snapshots; parse with SparseVector.parse()",
        "unknown_node": "(?) is a valid node representing unknown downstream services",
    }
}
 
stats_df = spark.createDataFrame([(json.dumps(stats, indent=2),)], ["json"])
stats_df.coalesce(1).write.mode("overwrite").text(f"{OUTPUT_ROOT}/dataset_stats")
 
print(json.dumps(stats, indent=2))
 


{
  "num_nodes": 111296,
  "num_node_features": 18,
  "num_edge_features": 2,
  "node_feature_names": [
    "cpu_rs",
    "cpu_rolling_mean",
    "cpu_rolling_std",
    "cpu_delta",
    "memory_rs",
    "memory_rolling_mean",
    "memory_rolling_std",
    "memory_delta",
    "t_sin",
    "t_cos",
    "HTTP_MCR_transformed",
    "consumerMQ_MCR_transformed",
    "consumerRPC_MCR_transformed",
    "providerRPC_MCR_transformed",
    "HTTP_RT_transformed",
    "consumerMQ_RT_transformed",
    "consumerRPC_RT_transformed",
    "providerRPC_RT_transformed"
  ],
  "edge_feature_names": [
    "rt_qt",
    "rt_abs_qt"
  ],
  "train_t_range": [
    0,
    649
  ],
  "val_t_range": [
    650,
    699
  ],
  "test_t_range": [
    700,
    1439
  ],
  "s3_root": "s3://university-research-s20426/tgnn_dataset",
  "notes": {
    "rpctype_ohe": "Stored as string in edge snapshots; parse with SparseVector.parse()",
    "unknown_node": "(?) is a valid node representing unknown downstream services"
  }
}


In [0]:
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## Step 9 — Sanity Check All Three Splits
 
# COMMAND ----------
 
for split, t_list in [("train", train_t), ("val", val_t), ("test", test_t)]:
 
    t0 = t_list[0]
 
    nodes = (
        spark.read
        .option("mergeSchema", "true")
        .option("basePath", f"{OUTPUT_ROOT}/snapshots/{split}/nodes")
        .parquet(f"{OUTPUT_ROOT}/snapshots/{split}/nodes/t_idx={t0}")
    )
 
    edges = (
        spark.read
        .option("mergeSchema", "true")
        .option("basePath", f"{OUTPUT_ROOT}/snapshots/{split}/edges")
        .parquet(f"{OUTPUT_ROOT}/snapshots/{split}/edges/t_idx={t0}")
    )
 
    n_nodes = nodes.count()
    n_edges = edges.count()
 
    print(f"\n=== {split.upper()} ===")
    print(f"  Snapshots : {len(t_list)}  (t={t_list[0]} → t={t_list[-1]})")
    print(f"  t={t0} nodes : {n_nodes:,}")
    print(f"  t={t0} edges : {n_edges:,}")
 
    if n_nodes == 0:
        print("  ⚠️  WARNING: zero nodes at first snapshot")
    if n_edges == 0:
        print("  ⚠️  WARNING: zero edges at first snapshot")


=== TRAIN ===
  Snapshots : 650  (t=0 → t=649)
  t=0 nodes : 96,358
  t=0 edges : 352,121

=== VAL ===
  Snapshots : 50  (t=650 → t=699)
  t=650 nodes : 96,307
  t=650 edges : 35,688

=== TEST ===
  Snapshots : 740  (t=700 → t=1439)
  t=700 nodes : 96,299
  t=700 edges : 454,770


In [0]:
print("Raw data max t_idx:", d_cg.agg(F.max("t_idx")).collect()[0][0])

In [0]:
# ── 1. Unique MS names (nodes) ─────────────────────────────────────────────
all_msnames = (
    res_train.select(F.col("msname").alias("node_key"))
    .union(res_val.select(F.col("msname").alias("node_key")))
    .union(res_test.select(F.col("msname").alias("node_key")))
    .union(cg_train.select(F.col("UM").alias("node_key")))
    .union(cg_train.select(F.col("DM").alias("node_key")))
    .union(cg_val.select(F.col("UM").alias("node_key")))
    .union(cg_val.select(F.col("DM").alias("node_key")))
    .union(cg_test.select(F.col("UM").alias("node_key")))
    .union(cg_test.select(F.col("DM").alias("node_key")))
    .distinct()
)
print(f"Total unique nodes (msname): {all_msnames.count():,}")

# ── 2. Node feature rows (MS-level aggregated per timestamp) ───────────────
res_ms_train = (
    res_train
    .groupBy("msname", "t_idx")
    .agg(F.mean("cpu_utilization").alias("cpu_mean"))  # just for counting
)
res_ms_val = (
    res_val.groupBy("msname", "t_idx")
    .agg(F.mean("cpu_utilization").alias("cpu_mean"))
)
res_ms_test = (
    res_test.groupBy("msname", "t_idx")
    .agg(F.mean("cpu_utilization").alias("cpu_mean"))
)

print(f"Node feature rows — train : {res_ms_train.count():,}")
print(f"Node feature rows — val   : {res_ms_val.count():,}")
print(f"Node feature rows — test  : {res_ms_test.count():,}")

# ── 3. Edge rows after deduplication + rt > 0 filter ──────────────────────
cg_clean_train = (
    cg_train
    .filter(F.col("rt") > 0)
    .groupBy("UM", "DM", "t_idx", "rpctype")
    .agg(F.mean("rt").alias("rt_mean"))  # just for counting
)
cg_clean_val = (
    cg_val
    .filter(F.col("rt") > 0)
    .groupBy("UM", "DM", "t_idx", "rpctype")
    .agg(F.mean("rt").alias("rt_mean"))
)
cg_clean_test = (
    cg_test
    .filter(F.col("rt") > 0)
    .groupBy("UM", "DM", "t_idx", "rpctype")
    .agg(F.mean("rt").alias("rt_mean"))
)

print(f"Edge rows — train : {cg_clean_train.count():,}")
print(f"Edge rows — val   : {cg_clean_val.count():,}")
print(f"Edge rows — test  : {cg_clean_test.count():,}")

# ── 4. Per-snapshot averages ───────────────────────────────────────────────
print(f"\nAvg nodes/snapshot — train : {res_ms_train.count() / 650:,.0f}")
print(f"Avg nodes/snapshot — val   : {res_ms_val.count() / 50:,.0f}")
print(f"Avg nodes/snapshot — test  : {res_ms_test.count() / 740:,.0f}")

print(f"\nAvg edges/snapshot — train : {cg_clean_train.count() / 650:,.0f}")
print(f"Avg edges/snapshot — val   : {cg_clean_val.count() / 50:,.0f}")
print(f"Avg edges/snapshot — test  : {cg_clean_test.count() / 740:,.0f}")

Total unique nodes (msname): 15,036
Node feature rows — train : 846,950
Node feature rows — val   : 65,150
Node feature rows — test  : 27,363
Edge rows — train : 2,553,801
Edge rows — val   : 292,151
Edge rows — test  : 136,657

Avg nodes/snapshot — train : 1,303
Avg nodes/snapshot — val   : 1,303
Avg nodes/snapshot — test  : 37

Avg edges/snapshot — train : 3,929
Avg edges/snapshot — val   : 5,843
Avg edges/snapshot — test  : 185
